# 📊 01. Exploratory Data Analysis & Inspection
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

This notebook inspects the raw dataset across **Source 1** (reference), **Source 2**, **Source 3**, and the **Ground Truth** match annotations.
Key goals:
1. Examine record counts, schemas, and missing value rates.
2. Characterize the **Ground Truth**: match multiplicity and **singleton distribution** (5.58% singletons).
3. Detect cross-dataset domain shifts (e.g. open-set country discovery in Test set).

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data.loader import load_tsv_records
from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Dir: {cfg['paths']['raw_train_dir']}")

## 1. Load Raw Training Samples
Inspect the first 5 records from each source table.

In [ ]:
raw_train_dir = PROJECT_ROOT / cfg["paths"]["raw_train_dir"]
s1_sample = load_tsv_records(raw_train_dir / "train_source1.tsv", nrows=5)
s2_sample = load_tsv_records(raw_train_dir / "train_source2.tsv", nrows=5)
s3_sample = load_tsv_records(raw_train_dir / "train_source3.tsv", nrows=5)
gt_sample = load_tsv_records(raw_train_dir / "train_ground_truth.tsv", nrows=5)

print("--- Source 1 Sample (Reference) ---")
display(s1_sample)

print("--- Ground Truth Sample ---")
display(gt_sample)

## 2. Inspect Dataset Summary Statistics & Profiling Report
Load the precomputed profiling metrics generated by the inspection pipeline.

In [ ]:
import json

report_path = PROJECT_ROOT / "artifacts" / "reports" / "data_inspection_report.json"
if report_path.exists():
    with open(report_path, encoding="utf-8") as f:
        profile = json.load(f)
    print(f"Inspected at: {profile.get('inspected_at')}")
    summary_df = pd.DataFrame(profile["datasets"]).T
    display(summary_df[["row_count", "column_count", "file_size_mb"]])
else:
    print("Run `scripts/01_inspect_data.py` to generate the complete profiling report.")

## 3. Ground Truth Structure: Matches & Singletons
In this challenge:
- **Singletons** have `matched_entity_ids == ""` (no matching record in S2/S3).
- True matches have comma-separated entity IDs from S2 and/or S3.

In [ ]:
gt_head = load_tsv_records(raw_train_dir / "train_ground_truth.tsv", nrows=50000)
gt_head["is_singleton"] = gt_head["matched_entity_ids"].str.strip() == ""
singleton_rate = gt_head["is_singleton"].mean()

print(f"Sample size: {len(gt_head):,}")
print(f"Singleton count: {gt_head['is_singleton'].sum():,} ({singleton_rate:.2%})")
print(f"Entities with matches: {(~gt_head['is_singleton']).sum():,} ({1-singleton_rate:.2%})")

## 4. Open-Set Country Discovery
Training data contains only `US` and `IN`. In test data, **France (`FR`)** appears in ~15% of records!

In [ ]:
interim_dir = PROJECT_ROOT / cfg["paths"]["interim_dir"]
if (interim_dir / "test_source1.parquet").exists():
    test_s1 = pd.read_parquet(interim_dir / "test_source1.parquet", columns=["country"])
    print("Test S1 Country Distribution:")
    display(test_s1["country"].value_counts(dropna=False, normalize=True).apply(lambda x: f"{x:.2%}"))